In [ ]:
import pandas as pd
import numpy as np
from Bio import SeqIO
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
from sklearn.metrics import confusion_matrix
import shap


In [ ]:
def run_the_classifier(data_df_merged_balanced):
    data_df_merged_balanced = data_df_merged_balanced.groupby('ecosystem_type').sample(n=541)
    # Import train_test_split function
    
    X=data_df_merged_balanced.iloc[:,0:data_df_numerical.columns.shape[0]] # Features
    y=data_df_merged_balanced['ecosystem_type']  # Labels
    y = pd.get_dummies(y, dtype=int)
    # Split dataset into training set and test set
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8,test_size=0.2,stratify=y) # 80% training and 20% test

    # Create a Gaussian Classifier
    # n_estimators is the number of the decision trees in the forest. Increasing number generally improves the performance of the model but also increases the computational cost
    clf=RandomForestClassifier(class_weight='balanced', max_features='log2',
                       min_samples_split=10, n_estimators=5000)

    #Train the model using the training sets y_pred=clf.predict(X_test)
    clf.fit(X_train,y_train)

    # test the model using the 30% test dataset
    y_pred=clf.predict(X_test)

    # get accuracy
    accuracy = metrics.accuracy_score(y_test, y_pred)
    feature_imp = pd.Series(clf.feature_importances_,index=data_df_numerical.columns).sort_values(ascending=False)

    shap_imp_df = calculate_shap_values(clf, X_test, X_train)

    return accuracy, feature_imp, shap_imp_df


In [ ]:
def calculate_shap_values(clf, X_test, X_train):
    # SHAP values
    
    # Create the SHAP explainer for the Random Forest model
    explainer = shap.TreeExplainer(clf)

    # Calculate SHAP values for the test set
    shap_values = explainer.shap_values(X_test)

    def shap_values_to_list(shap_values, clf):
        shap_as_list=[]
        for i in range(len(clf.classes_)):
            shap_as_list.append(shap_values[:,:,i])
        return shap_as_list

    shap_as_list = shap_values_to_list(shap_values, clf)
    
    feature_names = X_train.columns
    rf_resultX = pd.DataFrame(shap_as_list[0], columns = feature_names)

    vals = np.abs(rf_resultX.values).mean(0)

    shap_importance = pd.DataFrame(list(zip(feature_names, vals)),
                                    columns=['col_name','feature_importance_vals'])
    shap_importance = shap_importance.sort_values(by=['feature_importance_vals'],
                                ascending=False).set_index('col_name')

    return shap_importance


In [ ]:
fasta = 'foldmason_0_1_2_structure_MSA_aa_trimmed_gappyout.fa'

seq_dict = {} 

for record in SeqIO.parse(fasta, "fasta"):
    seq_dict[record.id]=record.seq

seq_dict
# convert to a dataframe
# data_df = pd.DataFrame.from_dict(seq_dict,orient='index')
# print(data_df)

# for the structural MSA
data_df = pd.DataFrame.from_dict(seq_dict,orient='index').reset_index(names='protein')
data_df.loc[data_df['protein'].str.contains('IMG'), 'img_split'] = (data_df.loc[data_df['protein'].str.contains('IMG'), 'protein']
      .str.split('|').str[0])
data_df['img_split'] = data_df['img_split'].fillna(data_df['protein'])
data_df.set_index('img_split', inplace=True)
data_df.drop('protein', axis=1, inplace=True)


data_df_numerical = pd.get_dummies(data_df, dtype=int)
data_df_numerical = data_df_numerical.loc[:,~data_df_numerical.columns.str.contains('-')]       # remove the position-gap columns
# print(data_df_numerical)

biomes_metadata_df = pd.read_csv('/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/amino_acid_analyis/imgvr_mgnify_combined/best_biome_per_mcp_protein.csv', header=0, sep=',')

# biomes_metadata_df
data_df_merged = data_df_numerical.join(biomes_metadata_df.set_index('protein'), how='left')
# print(data_df_merged)

temp = data_df_merged.groupby('ecosystem_type').count() # only to count the number of samples in each class before subsampling
# print(temp)

data_df_merged_balanced = data_df_merged[~data_df_merged['ecosystem_type'].isin(['Landfill','Non-marine Saline and Alkaline','Activated Sludge','Defined media','Digestive system','Industrial wastewater','Terephthalate','Water and sludge','Nutrient removal','Sediment','Estuary','Lentic','Aquaculture'])]


In [ ]:
data_df_merged_balanced

In [ ]:
data_df_merged_balanced.to_csv('data_df_merged_balanced_lake_hot_0_1_2.csv')

In [ ]:
# data_df_merged

In [ ]:
accuracy_dict = {}
importance_dict = {}

if __name__ == '__main__':
    for i in range(0, 50):
        accuracy, feature_imp, shap_imp_df = run_the_classifier(data_df_merged_balanced)
        accuracy_dict[i] = accuracy
        importance_dict[i] = feature_imp
        shap_imp_df = shap_imp_df



## Average feature importances by gini

In [ ]:
df_importances = pd.DataFrame.from_dict(importance_dict)
df_importances_mean = df_importances.mean(axis=1).sort_values(ascending=False)
df_importances_mean.head(n=20)

## average feature importance by shap

In [ ]:
df_shap_imp_mean = shap_imp_df.mean(axis=1).sort_values(ascending=False)
df_shap_imp_mean.head(n=20)

In [ ]:
feature_imp_merged_df = pd.concat([df_shap_imp_mean,df_importances_mean], axis=1)
feature_imp_merged_df = feature_imp_merged_df.rename({0: 'shap_imp', 1 : 'gini_imp'}, axis=1)
feature_imp_merged_df
feature_imp_merged_df.to_csv('caudo_50_rounds_MSTA_lake_hot_impscores_0_1_2.tsv',sep='\t')

## Average accuracy

In [ ]:
mean_accuracy = np.array(list(accuracy_dict.values())).mean()
mean_accuracy